# FNDE education contribution: how the raw layer is obtained

**Publisher:** Fundo Nacional de Desenvolvimento da Educação (FNDE)  
**Official source:** <https://www.fnde.gov.br/>

Collection and distribution of the education contribution (salário-educação) published by FNDE.

**Grain and keys:** See each table.

**Released as:**

- `fnde-salario-educacao-analytics`: semantic (2 tables)
- `fnde-salario-educacao-raw-trusted-part-1`: raw (40 tables)
- `fnde-salario-educacao-raw-trusted-part-2`: raw, trusted (6 tables)

Every table and column is documented in the [interactive data map](https://rangeltech.net/datamap/). Source code and the other notebooks: [https://github.com/LucasRangelSSouza/brazil-public-data-map](https://github.com/LucasRangelSSouza/brazil-public-data-map).

## Status of this notebook

**Not executed end to end.** The download cells below use URLs recorded in the released tables, but the publisher's download host did not respond when this notebook was written, so the outputs are empty. Run it from your own network.

In [ ]:
import json, time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; public-data-map-notebook; +https://github.com/LucasRangelSSouza/brazil-public-data-map)"}

def get(url, retries=4, timeout=90, **kw):
    """GET with exponential backoff. Public portals answer 429 or reset connections under load."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout, **kw)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

## 1. Which files make up the raw layer

FNDE publishes the education contribution as spreadsheets and PDFs on its site, one per year; each row of the released tables keeps the URL of the file it came from.

The released table `raw__fnde_salario_educacao_realizado_2024` lists every file with its page and download URL, so the list below is the record of what was fetched.

In [ ]:
import kagglehub, pandas as pd
listing = pd.read_parquet(kagglehub.dataset_download("lucasrangelss/fnde-salario-educacao-raw-trusted-part-2", path="raw__fnde_salario_educacao_realizado_2024.parquet"))
listing[[c for c in ["source_url", "source_url"] if c in listing.columns]].drop_duplicates().head(10)

## 2. Download one file

The publisher's download host sometimes resets connections. The helper retries with backoff and a browser-like User-Agent. Pick the smallest file first.

In [ ]:
from pathlib import Path
import zipfile

url = listing["source_url"].dropna().iloc[0]
target = Path("downloads") / url.rsplit("/", 1)[-1]
target.parent.mkdir(exist_ok=True)
try:
    with get(url, stream=True) as r, open(target, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
    print("saved", target, target.stat().st_size, "bytes")
    if target.suffix == ".zip":
        print(zipfile.ZipFile(target).namelist()[:10])
except Exception as exc:
    print("download failed:", exc)

## 3. What the raw layer does with it

The raw layer keeps the spreadsheet or CSV columns as delivered (names and strings), adds the source file name and removes ingestion bookkeeping. Typing, deduplication and renaming happen in `trusted`.

## Compare with the released tables

The released files keep the field names of the source in `raw` and apply consistent typing and snake_case names in `trusted`. Details per column are in the [data map](https://rangeltech.net/datamap/#/subject/fnde-salario-educacao).

In [ ]:
# Read one released table without downloading the whole dataset. Public datasets need no login.
import kagglehub, pandas as pd
path = kagglehub.dataset_download("lucasrangelss/fnde-salario-educacao-raw-trusted-part-2", path="trusted__fnde_salario_educacao_previsto.parquet")
pd.read_parquet(path).head()